In [0]:
nhs_suppliers_df = spark.read.table('my_catalog.bsheffield_spark.nhs_suppliers')
nhs_contract_description = spark.read.table('my_catalog.bsheffield_spark.nhs_contract_description')
nhs_trusts = spark.read.table('my_catalog.bsheffield_spark.nhs_trusts')
nhs_deliveries = spark.read.table('my_catalog.bsheffield_spark.nhs_deliveries')
nhs_costs_df = spark.read.table('my_catalog.bsheffield_spark.contracts_cost')

In [0]:
nhs_combo_df = nhs_suppliers_df \
    .select('supplier_id', 'supplier_name') \
    .join(nhs_contract_description, on='supplier_id', how='inner') \
    .join(nhs_trusts.select('trust_id', 'trust_name'), on='trust_id', how='inner') \
    .join(nhs_deliveries, on='contract_id', how='inner') \
    .join(nhs_costs_df, on='contract_id', how='inner')

In [0]:
nhs_combo_df_pandas = nhs_combo_df.toPandas()

# Save as Spark table for downstream tasks
spark_df = spark.createDataFrame(nhs_combo_df_pandas)
spark_df.write.mode("overwrite").saveAsTable("my_catalog.bsheffield_spark.nhs_combined_dataset")


In [0]:
# For LATER APP Interactions 
contract_templates = nhs_combo_df_pandas[['contract_id', 'contract_type', 'supplier_name', 'postcode', 'issue_date', 'closing_date']].drop_duplicates()

spark.sql("DROP TABLE IF EXISTS my_catalog.bsheffield_spark.contract_templates")
spark.createDataFrame(contract_templates).write.saveAsTable("my_catalog.bsheffield_spark.contract_templates")


In [0]:
contract_templates

,contract_id,contract_type,supplier_name,postcode,issue_date,closing_date
0,NHS-CT-00001,Sharps Disposal,MedSupply UK,L9,2024-12-06,2024-12-31
1,NHS-CT-00002,Reagent Bundles,BioLogica Ltd,FY3,2024-10-08,2024-10-19
2,NHS-CT-00003,PCR Test Kits,BioLogica Ltd,L9,2025-03-30,2025-04-25
3,NHS-CT-00004,Sharps Disposal,SterileFlow,FY1,2024-04-02,2024-04-24
4,NHS-CT-00005,Reagent Bundles,HealthLab Partners,PR2,2025-01-01,2025-01-17
...,...,...,...,...,...,...
995,NHS-CT-00996,Reagent Bundles,BioLogica Ltd,PR1,2024-12-09,2024-12-22
996,NHS-CT-00997,Blood Collection Kits,SterileFlow,WA7,2025-03-19,2025-04-14
997,NHS-CT-00998,Sharps Disposal,MedSupply UK,PR2,2024-12-19,2024-12-29
998,NHS-CT-00999,Sharps Disposal,SterileFlow,WA8,2024-09-16,2024-10-11
